# AvisSense — Rapport de métriques

Ce notebook calcule et affiche les graphiques de métriques du modèle, et les sauvegarde en PNG dans `reports/figures/` (prêts à insérer dans les slides / la vidéo) :

1. **Courbes de perte** train/validation par époque (diagnostic surapprentissage)
2. **Accuracy et F1 de validation** par époque
3. **Matrice de confusion** en valeurs absolues
4. **Matrice de confusion** en % par classe réelle
5. **Baseline (modèle brut) vs fine-tuné** : le gain apporté par le transfer learning

**Utilisation** :
- *En local* : ouvrir ce notebook depuis la racine du projet (le modèle doit être entraîné dans `model/sentiment_model`).
- *Sur Google Colab* : exécuter toutes les cellules dans l'ordre — le repo est cloné et un entraînement rapide est lancé automatiquement si aucun modèle n'est présent.

Toute la logique de tracé vit dans `scripts/generate_metrics_plots.py` (source unique) : le notebook ne fait que l'appeler et afficher les images.

In [ ]:
# Installation des dépendances (utile sur Colab ; sans effet si déjà installées)
%pip install -q transformers datasets accelerate scikit-learn matplotlib torch

In [ ]:
# Se placer à la racine du projet.
# - En local : le notebook est dans notebooks/, on remonte d'un cran.
# - Sur Colab : on clone le repo s'il n'est pas déjà là.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")                      # local : notebooks/ -> racine
elif not Path("scripts").exists():
    !git clone https://github.com/MEMS17/AvisSense.git
    os.chdir("AvisSense")               # Colab : on entre dans le clone

PROJECT_ROOT = Path.cwd()
print(f"Racine du projet : {PROJECT_ROOT}")

In [ ]:
# Entraînement rapide SI aucun modèle n'est présent (sinon cette cellule ne fait rien).
# Config réduite : ~10 min sur le GPU gratuit de Colab. Pour le run complet,
# retirer les options (--max-train 8000 par défaut).
if not Path("model/sentiment_model").exists():
    !python scripts/train.py --max-train 2000 --max-val 500 --max-test 500 --epochs 2
else:
    print("Modèle déjà présent dans model/sentiment_model — entraînement sauté.")

In [ ]:
# Imports : on réutilise les fonctions du script (source unique de la logique de tracé)
import sys

sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from datasets import load_dataset
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from scripts.generate_metrics_plots import (
    BASE_MODEL,
    FIGURES_DIR,
    MODEL_DIR,
    SEED,
    load_history,
    plot_baseline_comparison,
    plot_confusion,
    plot_epoch_metrics,
    plot_loss_curves,
    predict_in_batches,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Matériel : {device}")

## 1-2. Courbes d'entraînement

Lecture : la **training loss doit descendre** au fil des époques. Si la **validation loss remonte** pendant que la training loss continue de descendre, le modèle mémorise le train au lieu de généraliser : c'est le **surapprentissage** (overfitting). Les métriques de validation (accuracy, F1) doivent rester stables ou monter.

In [ ]:
history = load_history()
if history:
    plot_loss_curves(history, FIGURES_DIR / "1_loss_curves.png")
    plot_epoch_metrics(history, FIGURES_DIR / "2_metrics_par_epoque.png")
    display(Image(str(FIGURES_DIR / "1_loss_curves.png")))
    display(Image(str(FIGURES_DIR / "2_metrics_par_epoque.png")))
else:
    print("training_history.json absent : relancez scripts/train.py (le fichier\n"
          "est écrit depuis la mise à jour du script ; les anciens runs ne l'ont pas).")

## 3-4. Matrices de confusion sur le test set

Le test set n'a servi ni à entraîner ni à choisir le meilleur checkpoint : c'est la mesure honnête. Lecture : la **diagonale** = les bonnes réponses ; hors diagonale = les erreurs. La version en **%** (normalisée par classe réelle) montre si le modèle est meilleur sur une classe que sur l'autre.

In [ ]:
MAX_TEST = 2000      # Réduire (ex : 500) pour aller plus vite sur CPU
BATCH_SIZE = 32

# Chargement du test set (échantillon aléatoire reproductible, seed fixe)
test_data = load_dataset("allocine", split="test").shuffle(seed=SEED)
test_data = test_data.select(range(min(MAX_TEST, len(test_data))))
texts = test_data["review"]
labels = np.array(test_data["label"])

# Prédictions du modèle fine-tuné
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(device)
predictions = predict_in_batches(texts, model, tokenizer, BATCH_SIZE, device)

finetuned_scores = [accuracy_score(labels, predictions), f1_score(labels, predictions)]
print(f"\nFine-tuné : accuracy={finetuned_scores[0]:.4f} | f1={finetuned_scores[1]:.4f}")

matrix = confusion_matrix(labels, predictions)
plot_confusion(matrix, FIGURES_DIR / "3_confusion_matrix_abs.png", as_percent=False)
plot_confusion(matrix, FIGURES_DIR / "4_confusion_matrix_pct.png", as_percent=True)
display(Image(str(FIGURES_DIR / "3_confusion_matrix_abs.png")))
display(Image(str(FIGURES_DIR / "4_confusion_matrix_pct.png")))

## 5. Baseline vs fine-tuné — le gain du transfer learning

La **baseline** est le même DistilCamemBERT mais avec sa tête de classification **non entraînée** (initialisée aléatoirement) : elle répond au hasard, ~50 % sur 2 classes équilibrées. L'écart entre les deux barres = ce que le fine-tuning a réellement apporté.

> Le warning `Some weights were newly initialized` est **normal** ici : c'est précisément le principe de la baseline.

In [ ]:
baseline_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2
).to(device)
baseline_predictions = predict_in_batches(texts, baseline_model, baseline_tokenizer, BATCH_SIZE, device)

baseline_scores = [accuracy_score(labels, baseline_predictions),
                   f1_score(labels, baseline_predictions)]
print(f"\nBaseline : accuracy={baseline_scores[0]:.4f} | f1={baseline_scores[1]:.4f}")
print(f"Fine-tuné : accuracy={finetuned_scores[0]:.4f} | f1={finetuned_scores[1]:.4f}")

plot_baseline_comparison(baseline_scores, finetuned_scores,
                         FIGURES_DIR / "5_baseline_vs_finetune.png")
display(Image(str(FIGURES_DIR / "5_baseline_vs_finetune.png")))

## Récupérer les images

Les 5 figures sont sauvegardées dans **`reports/figures/`** :

```
reports/figures/
├── 1_loss_curves.png
├── 2_metrics_par_epoque.png
├── 3_confusion_matrix_abs.png
├── 4_confusion_matrix_pct.png
└── 5_baseline_vs_finetune.png
```

- *En local* : le dossier est à la racine du projet.
- *Sur Colab* : panneau **Fichiers** (icône dossier à gauche) → `AvisSense/reports/figures/` → clic droit → *Télécharger*. Ou en une commande :

```python
from google.colab import files
!zip -r figures.zip reports/figures
files.download("figures.zip")
```

Équivalent en ligne de commande (sans notebook) : `python scripts/generate_metrics_plots.py`